In [1]:
import numpy as np

#### Using the Spherical Law of Cosines, convert the inclination w.r.t. the ecliptic to the inclination e.r.t. the Sun's Equator

The formula to find the inclination to the Sun's equator ($i_{eq}$) is:$$\cos(i_{eq}) = \cos(i_{ec}) \cos(i_{\odot}) + \sin(i_{ec}) \sin(i_{\odot}) \cos(\Omega - \Omega_{\odot})$$

In [2]:

class Asteroid_Table:
    # Define the constants (in degrees)

    # The inclination of the Sun's equator relative to the ecliptic:
    i_sun = 7.25
    # The Longitude of the Ascending Node of the Sun's equator 
    omega_sun = 75.76

    def __init__(self, name, semi_major_axis, eccentricity, inclination_ecliptic, longitude_of_ascending_node):
        self.name = name
        self.a = semi_major_axis
        self.e = eccentricity
        self.i_ec = inclination_ecliptic 
        self.node = longitude_of_ascending_node 

    def get_i_eq(self):
        """
        Calculates the inclination with respect to the Sun's equator.
        Handles the necessary conversions between degrees and radians.
        """
        # Step 1. Convert all degree values to radians for numpy math
        i_ec_rad = np.radians(self.i_ec)
        i_sun_rad = np.radians(self.i_sun)
        node_rad = np.radians(self.node)
        omega_sun_rad = np.radians(self.omega_sun)
        
        # Step 2. Apply The Spherical Law of Cosines
        cos_i_eq = (np.cos(i_ec_rad) * np.cos(i_sun_rad) + 
                    np.sin(i_ec_rad) * np.sin(i_sun_rad) * np.cos(node_rad - omega_sun_rad))
        
        # Step 3. Apply the arccosine and convert back to degrees
        i_eq_rad = np.arccos(cos_i_eq)
        return np.degrees(i_eq_rad)

# Orbital elements populated from Jet Propulsion Laboratory, Solar System Database (https://ssd.jpl.nasa.gov/horizons/app.html#/)
# Osculating Epoch: [Whatever date was on the site], Reference Frame: ECLIPJ2000

# Format: Name, a (au), e, i_ec (degrees), node (degrees)
asteroids = [
    Asteroid_Table("1566 Icarus", 1.078, 0.827, 22.8254, 88.0107),
    Asteroid_Table("1998 TU3",    0.787, 0.484, 5.4132,  127.0503),
    Asteroid_Table("1999 KW4",    0.642, 0.688, 38.8839, 244.9084),
    Asteroid_Table("1999 MN",     0.674, 0.665, 2.0163,  71.0427),
    Asteroid_Table("2000 BD19",   0.876, 0.895, 25.6882, 333.6548),
    Asteroid_Table("2000 EE14",   0.662, 0.533, 26.5186, 283.0031),
    Asteroid_Table("2001 YE4",    0.677, 0.541, 4.8217,  108.9712),
    Asteroid_Table("2004 KH17",   0.712, 0.499, 22.1009, 22.1818),
    Asteroid_Table("2006 CJ",     0.676, 0.755, 10.2526, 57.0560)
]

# Then print the formatted table header
print(f"{'Target':<15} | {'a (au)':<10} | {'e':<10} | {'i_ec (deg)':<20} | {'i_eq (deg)':<20}")
print("-" * 85)

# Calculate and print the unrounded values for each asteroid 
for ast in asteroids:
    i_eq_unrounded = ast.get_i_eq()
    print(f"{ast.name:<15} | {ast.a:<10} | {ast.e:<10} | {ast.i_ec:<20} | {i_eq_unrounded:<20}")

Target          | a (au)     | e          | i_ec (deg)           | i_eq (deg)          
-------------------------------------------------------------------------------------
1566 Icarus     | 1.078      | 0.827      | 22.8254              | 15.811537342448243  
1998 TU3        | 0.787      | 0.484      | 5.4132               | 5.716948122024405   
1999 KW4        | 0.642      | 0.688      | 38.8839              | 46.02121295369019   
1999 MN         | 0.674      | 0.665      | 2.0163               | 5.243138767511647   
2000 BD19       | 0.876      | 0.895      | 25.6882              | 28.055839492951286  
2000 EE14       | 0.662      | 0.533      | 26.5186              | 33.11878914141162   
2001 YE4        | 0.677      | 0.541      | 4.8217               | 4.156861455781174   
2004 KH17       | 0.712      | 0.499      | 22.1009              | 18.68292965448669   
2006 CJ         | 0.676      | 0.755      | 10.2526              | 4.099908527963058   


In [3]:
# The time when the initial conditions were extracted:



### Create a CSV File with the initial conditions (precise measurements) for the 9 asteroids (by pulling them from the SSD JPL NASA website)

In [4]:
import requests
import pandas as pd


def fetch_asteroid_full_precision(name):
    url = "https://ssd-api.jpl.nasa.gov/sbdb.api"
    params = {"sstr": name, "full-prec": "true"}

    data = requests.get(url, params=params).json()
    elems = data["orbit"]["elements"]

    def get(label):
        for el in elems:
            if el.get("label") == label:
                return el.get("value")
        return None

    return {
        "name": name,
        "a": get("a"),
        "e": get("e"),
        "i": get("i"),
        "node": get("node"),
        "peri": get("peri"),
        "M": get("M"),
        "n": get("n"),
        #"epoch": get("epoch")
    }


asteroid_names = [
    "1566 Icarus",
    "1998 TU3",
    "1999 KW4",
    "1999 MN",
    "2000 BD19",
    "2000 EE14",
    "2001 YE4",
    "2004 KH17",
    "2006 CJ"
]

rows = [fetch_asteroid_full_precision(n) for n in asteroid_names]

df = pd.DataFrame(rows)

df.to_csv("asteroids_full_precision.csv", index=False)

print("CSV written: asteroids_full_precision.csv")

CSV written: asteroids_full_precision.csv


In [ ]:
import numpy as np
import pandas as pd


class Asteroid_Table:
    # Define the constants (in degrees)

    # The inclination of the Sun's equator relative to the ecliptic:
    i_sun = 7.25
    # The Longitude of the Ascending Node of the Sun's equator 
    omega_sun = 75.76

    def __init__(self, name, a, e, i_ec, node, peri, M, n):
        self.name = name

        # Store the values as floats for computation
        self.a = float(a)
        self.e = float(e)
        self.i_ec = float(i_ec)
        self.node = float(node)
        self.peri = float(peri)
        self.M = float(M)
        self.n = float(n)
        #self.epoch = epoch  

    def get_i_eq(self):
        """
        A function to calculate the inclination with respect to the Sun's equator.
        The function also tackles the necessary conversions between degrees and radians.
        """

        # Step 1. Convert all degree values to radians for numpy math
        i_ec_rad = np.radians(self.i_ec)
        i_sun_rad = np.radians(self.i_sun)
        node_rad = np.radians(self.node)
        omega_sun_rad = np.radians(self.omega_sun)

        # Step 2. Apply The Spherical Law of Cosines
        cos_i_eq = (
            np.cos(i_ec_rad) * np.cos(i_sun_rad) + np.sin(i_ec_rad) * np.sin(i_sun_rad)
            * np.cos(node_rad - omega_sun_rad)
        )

        # Step 3. Apply the arccosine and convert back to degrees
        i_eq_rad = np.arccos(cos_i_eq)
        return np.degrees(i_eq_rad)


# Now pull out the asteroids' values from the CSV file
def load_asteroids(csv_file):
    df = pd.read_csv(csv_file)

    return [
        Asteroid_Table(
            row["name"],
            row["a"],
            row["e"],
            row["i"],
            row["node"],
            row["peri"],
            row["M"],
            row["n"],
            #row["epoch"]
        )
        for _, row in df.iterrows()
    ]


asteroids = load_asteroids("asteroids_full_precision.csv")


print(
    f"{'Target':<15} | {'a (AU)':<8} | {'e':<8} | {'i (deg)':<8} | {'node (deg)':<10} | "
    f"{'peri (deg)':<10} | {'M (deg)':<10} | {'i_eq (deg)':<8} | {'n (deg/day)':<8}"
)

print("-" * 120)

for ast in asteroids:
    i_eq = ast.get_i_eq()

    print(
        f"{ast.name:<15} | "
        f"{ast.a:<8.5f} | "
        f"{ast.e:<8.5f} | "
        f"{ast.i_ec:<8.5f} | "
        f"{ast.node:<10.5f} | "
        f"{ast.peri:<10.5f} | "
        f"{ast.M:<10.5f} | "
        #f"{ast.epoch:<20} | "
        f"{i_eq:<10f} |"
        f"{ast.n:<10.5f} | "       
    )

Target          | a (AU)   | e        | i (deg)  | node (deg) | peri (deg) | M (deg)    | i_eq (deg) | n (deg/day)
------------------------------------------------------------------------------------------------------------------------
1566 Icarus     | 1.07799  | 0.82702  | 22.80164 | 87.94856   | 31.44439   | 329.18266  | 15.785529  |0.88060    | 
1998 TU3        | 0.78757  | 0.48370  | 5.41528  | 101.87731  | 84.98796   | 356.82196  | 3.369478   |1.41017    | 
1999 KW4        | 0.64239  | 0.68838  | 38.87862 | 244.89559  | 192.65394  | 5.45674    | 46.015670  |1.91429    | 
1999 MN         | 0.67407  | 0.66532  | 2.03184  | 81.38441   | 9.20970    | 103.70942  | 5.231713   |1.78091    | 
2000 BD19       | 0.87648  | 0.89497  | 25.75726 | 333.65517  | 324.38756  | 213.42344  | 28.122440  |1.20114    | 
2000 EE14       | 0.66183  | 0.53307  | 26.45905 | 155.74476  | 197.85461  | 94.64671   | 26.119959  |1.83055    | 
2001 YE4        | 0.67685  | 0.54043  | 4.85329  | 305.31620  | 319.

## How to calculate the True Anomaly (with Newton-Raphson)

In [6]:
from math import pi
import numpy as np
from numpy import log as ln

Given e and M, you can also solve for Kepler’s equation:

$$ f(E) = E − e sin(E) − M = 0 $$

where M is the mean anomaly, e is the eccentricity, and E is the eccentric anomaly. Know that e = 0.02 is like the Earth, and e = 0.9 is like a highly eccentric planet.

---
As the eccentric anomaly is to be plotted as a function of the mean anomaly, then define the eccentric anomaly as:

$$ M = E - e sin(E) $$

Additionally note that the mean anomaly is:

$$ M = \frac{2\pi}{T} (t - t_0)$$

And to solve this problem solve for when f(E) = 0.

---

To solve this exercise, do the same as for Q3b, and therefore know that the Newton-Raphson iteration formula is:

$$E_{i + 1} = E_{i} - \frac{f(E_i)}{f'(E_i)} = E_{i} - \frac{E - e \cdot sin(E) - M}{1 - e \cdot cos(E)}$$

In [7]:
def root_finder(E_i_in, M_in, e_in):
    """
    A function which return the eccentric anomaly
    given a mean anomaly and eccentricity
    """
    
    #Define the function
    f_Q3c = lambda E, M, e: E - e*np.sin(E) - M
    
    #Define the derivative
    f_Q3c_der = lambda E, e: 1 - e*np.cos(E)
    
    #Assume we will need less than 100 iterations
    iterations_in = 100
    
    N_array = np.array(range(iterations_in))
    
    for n in N_array:

        #Defining the iteration formula
        E_iplus1_in = E_i_in - (f_Q3c(E_i_in, M_in, e_in))/(f_Q3c_der(E_i_in, e_in))
        #print(E_iplus1_in, E_i_in)

        if abs(E_iplus1_in - E_i_in) < 10**(-15): #It converges to this number very quickly (within 5 itterations)
            #Beware if it doesn't converge quickly, then it may run until 100
            return E_iplus1_in

        E_i_in = E_iplus1_in
        
    return E_iplus1_in

Then once as we've found E, we can now compute $\nu$, the true anomaly:

$ \nu = 2 \cdot arctan(\sqrt{\frac{1+e}{1-e}} tan \frac{E}{2}) $


In [8]:
def nu_function(e_in, E_in):

    nu_out = 2 * np.arctan( np.sqrt((1+e_in)/(1-e_in)) * np.tan(E_in/2))

    return nu_out

In [9]:
eccentric_anomaly_list = []
true_anomaly_list = []

for ast in asteroids:
    i_eq = ast.get_i_eq()

    # The mean anomaly (convert to radians)
    M_rad = np.radians(ast.M)

    # Make an educated guess for E_in
    #I assume it's similar to M (mean anomaly)
    E_ast = root_finder(M_rad, M_rad, ast.e)
    eccentric_anomaly_list.append(E_ast)

    nu_ast = nu_function(ast.e, E_ast)
    true_anomaly_list.append(nu_ast)

print(eccentric_anomaly_list)
print(true_anomaly_list)



[4.939548539638194, 6.175945899165832, 0.2961089285167313, 2.304293601992018, 3.4517711221639193, 2.109478015673329, 3.2225678031126663, 1.8734036118211002, 0.6519265450319258]
[-2.4032012135854846, -0.18146697915286625, 0.6682952683598539, 2.7478161669333074, -3.0680106050189173, 2.5347260893974632, -3.097346647190208, 2.3367349966635618, 1.4705353041856568]


In [10]:
# To make sure the values are correct:

#For TU3 the mean anomaly (M) is 74.76747570797549 (in degrees)
M_TU3 = np.radians(74.76747570797549)

#The eccentricity is unitless (describes how stretchy the surface is)
e_TU3 = 0.4836694929440215

#Make an educated guess
#Additionally, as M and e are single values, E can also be one initial guess
#I assume it's similar to M_TU3
E_value = M_TU3

# Find the eccentric anomaly
E_TU3 = root_finder(E_value, M_TU3, e_TU3)
print(E_TU3)

#The true anomaly:
print(f"{nu_function(e_TU3, E_TU3):.10e}")

1.7782386710231803
2.2486890775e+00


## Determining the Yarkovksy Effect for the 9 asteroids

In [29]:
# For TU3 the value of A2 is computed 

avg_a_TU3_greenberg = -5.60 * 10**(-4)                       # AU / Myr 
avg_a_TU3_calc = -5.60 * 10**(-4) / (10**6 * 365.25)            # AU / days
uncertain_avg_a_TU3_greenberg = 3.9 * 10**(-4)               # AU / Mry
uncertain_avg_a_TU3_calc = 3.9 * 10**(-4) / (10**6 * 365.25)    # AU / days

# avg_a_greenberg_list = [ -4.84, -560, ]
# Icarus, TU3, 

a_TU3_calc = 0.7875484323220899 # AU
e_TU3_calc = 0.4836694929440215 # unitless
n_TU3_calc = 1.410224212386279  # degrees/ days (remember that angles are unitless in unit conversion)

A_2_TU3_AUdays2 = avg_a_TU3_calc * (n_TU3_calc * (a_TU3_calc**2) * (1 - (e_TU3_calc**2))) / 2
A_2_TU3_uncertainty_AUdays2 = uncertain_avg_a_TU3_calc * (n_TU3_calc * (a_TU3_calc**2) * (1 - (e_TU3_calc**2))) / 2

print(f"The Yarkovksy parameter for TU3: {A_2_TU3_AUdays2} AU / days^2")
print(A_2_TU3_uncertainty_AUdays2)


The Yarkovksy parameter for TU3: -5.136596706599691e-13 AU / days^2
3.5772727063819274e-13


In [30]:
# target_mpc_code = [1566, 66146, 66391, 437844, 137924, 138127, 480883, 468468, 364136]
# I could not find 2000 EE14 (the 6th asteroid...)
avg_a_greenberg = np.array([-4.74, -1.60, -5.23, 37.93, -22.10, 0, -48.47, -43.80,  -24.97]) 
avg_a_greenberg_uncertainty = np.array([0.4, 4.5, 2.9, 3.9, 13.5, 0,  2.0, 5.3, 10.3]) 

avg_a_greenberg_r = np.array([-4.84, -5.60, -5.73, 37.42, -26.17, 0, -49.84, -43.81, -38.23]) 
avg_a_greenberg_uncertainty_r = np.array([0.4, 3.9, 2.20, 3.8, 10.4, 0,  0.7, 5.3, 1.8]) 

In [31]:
def Yarkovsky_ms2(avg_a_in, a_in, e_in, n_in):

    """
    A function to calculate the Yarkovksy Parameter.
    The inputs should be <da/dt> in the units of * 10**(-4) AU/Myr, as they appear in Greenberg 2020, 
    semi-axis (a) in AU, eccentricity (unitless), and true anomaly (degrees/day).

    The calculations in this function are described when discussing the Yarkovksy Effect in the Bachelor Thesis.
    """

    # Convert from AU / Myr to AU / days
    avg_a_fn_calc = avg_a_in * 10**(-4) / (10**6 * 365.25)

    # Calculate the parameter
    A_2_fn_AUdays2 = avg_a_fn_calc * (n_in * (a_in**2) * (1 - (e_in**2))) / 2
    # Convert to m/s2, because Tudat requires this input
    A_2_fn_ms2 = A_2_fn_AUdays2 * 149597870691 / ((24 * 3600)**2)

    return A_2_fn_ms2

# -----------------------------------------------

A_2_values_list = []
i = -1

for ast in asteroids:
    i += 1
    i_eq = ast.get_i_eq()

    # print(
    #     f"{ast.name:<15} | "
    #     f"{ast.a:<8.5f} | "
    #     f"{ast.e:<8.5f} | "
    #     f"{ast.i_ec:<8.5f} | "
    #     f"{ast.node:<10.5f} | "
    #     f"{ast.peri:<10.5f} | "
    #     f"{ast.M:<10.5f} | "
    #     #f"{ast.epoch:<20} | "
    #     f"{i_eq:<10f} |"
    #     f"{ast.n:<10.5f} | "        # testing here
    # )
    A_2_ast_ms2 = Yarkovsky_ms2(avg_a_greenberg_r[i], ast.a, ast.e, ast.n)
    A_2_values_list.append(A_2_ast_ms2)

print(A_2_values_list)
AU_days2_to_m_s2 = (149597870691) / ((24 * 3600)**2)
print(AU_days2_to_m_s2)
A_2_values_AUdays2 = np.array(A_2_values_list) / 20 # * 10**(14)
print(A_2_values_AUdays2)


[-4.294149533646245e-12, -1.0293515312050457e-11, -6.53320849612292e-12, 4.6298801129934135e-11, -1.3184689979070076e-11, 0.0, -7.848755788425946e-11, -7.51616964714228e-11, -3.657371284005476e-11]
20.040009684043852
[-2.14707477e-13 -5.14675766e-13 -3.26660425e-13  2.31494006e-12
 -6.59234499e-13  0.00000000e+00 -3.92437789e-12 -3.75808482e-12
 -1.82868564e-12]
